# Using ThermoMPNN-D Data

## Download the dataset from Zenodo (if not downloaded already)

In [2]:
from pathlib import Path
import tarfile
from zenodo_get import download

DATA_DIR = Path("../../data/raw")
MEGASCALE_DIR = DATA_DIR / "megascale"


def get_megascale():
    if MEGASCALE_DIR.exists() and any(MEGASCALE_DIR.iterdir()):
        return MEGASCALE_DIR

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    archive = DATA_DIR / "Megascale.tar.gz"

    download(
        record_or_doi="13345274",
        output_dir=DATA_DIR,
        file_glob="Megascale.tar.gz",
    )

    MEGASCALE_DIR.mkdir(exist_ok=True)

    with tarfile.open(archive, "r:gz") as tar:
        for member in tar.getmembers():
            if member.name.startswith("Megascale/"):
                member.name = member.name.removeprefix("Megascale/")
                tar.extract(member, MEGASCALE_DIR)

    archive.unlink()

    return MEGASCALE_DIR


megascale_dir = get_megascale()

2026-09-29 13:08:13.306 | INFO     | zenodo_get.workflow.run_download:run_download:49 - Output directory: /home/jikael/uoft/research/mutation-prediction/data/raw
2026-09-29 13:08:13.986 | INFO     | zenodo_get.workflow.log_record_summary:log_record_summary:17 - Title: Double mutant stability data and single mutant structures for ThermoMPNN-D paper
2026-09-29 13:08:13.986 | INFO     | zenodo_get.workflow.log_record_summary:log_record_summary:24 - Total size: 34.4 MB
2026-09-29 13:08:13.986 | INFO     | zenodo_get.workflow.log_record_summary:log_record_summary:25 - Number of files: 1
2026-09-29 13:08:36.758 | SUCCESS  | zenodo_get.workflow.download_files:download_files:64 - All specified files have been processed.
/tmp/ipykernel_23289/2688576640.py:29: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, MEGASCALE_DIR)


Now that it's downloaded we can inspect the data:

| Column        | Value               | Meaning                                                                  |
| ------------- | ------------------- | ------------------------------------------------------------------------ |
| `aa_seq`      | `GATAVSEW...`       | The **full amino-acid sequence after both mutations**                    |
| `mut_type`    | `T9Q:N23Q`          | The two mutations: **T9 → Q** at position 9 and **N → Q** at position 23 |
| `WT_name`     | `1E0L.pdb`          | The **wild-type protein structure** being mutated                        |
| `ddG_ML`      | `-0.0137`           | Measured change in stability for the **double mutant**                   |
| `mut1`        | `T9Q`               | First mutation                                                           |
| `mut2`        | `N23Q`              | Second mutation                                                          |
| `description` | `1E0L.pdb_T9Q:N23Q` | Convenient identifier combining protein + mutations                      |


In [4]:
import pandas as pd

train = pd.read_csv(MEGASCALE_DIR / "csv/Megascale-D-train.csv")

train.head()

,Unnamed: 0,aa_seq,mut_type,WT_name,ddG_ML,mut1,mut2,description
0,0,GATAVSEWQEYKTADGKTYYYNQRTLESTWEKPQELK,T9Q:N23Q,1E0L.pdb,-0.013667,T9Q,N23Q,1E0L.pdb_T9Q:N23Q
1,1,GATAVSEWQEYKTADGKTYYYNERTLESTWEKPQELK,T9Q:N23E,1E0L.pdb,0.259309,T9Q,N23E,1E0L.pdb_T9Q:N23E
2,2,GATAVSEWQEYKTADGKTYYYNHRTLESTWEKPQELK,T9Q:N23H,1E0L.pdb,-0.658436,T9Q,N23H,1E0L.pdb_T9Q:N23H
3,3,GATAVSEWQEYKTADGKTYYYNSRTLESTWEKPQELK,T9Q:N23S,1E0L.pdb,-0.426091,T9Q,N23S,1E0L.pdb_T9Q:N23S
4,4,GATAVSEWQEYKTADGKTYYYNARTLESTWEKPQELK,T9Q:N23A,1E0L.pdb,-0.204260,T9Q,N23A,1E0L.pdb_T9Q:N23A


Get a list of all the `.pdb` (Protein Data Bank) files:

In [8]:
pdb_files = list(MEGASCALE_DIR.rglob("*.pdb"))
pdb_files

[PosixPath('../../data/raw/megascale/pdb/3MYE.pdb'),
 PosixPath('../../data/raw/megascale/pdb/2N4V.pdb'),
 PosixPath('../../data/raw/megascale/pdb/HHH_rd2_0139.pdb'),
 PosixPath('../../data/raw/megascale/pdb/v2_2BN8.pdb'),
 PosixPath('../../data/raw/megascale/pdb/7NIP.pdb'),
 PosixPath('../../data/raw/megascale/pdb/r6_859_TrROS_Hall.pdb'),
 PosixPath('../../data/raw/megascale/pdb/2L4J.pdb'),
 PosixPath('../../data/raw/megascale/pdb/2LHG.pdb'),
 PosixPath('../../data/raw/megascale/pdb/HHH_rd1_0210.pdb'),
 PosixPath('../../data/raw/megascale/pdb/EA:run7_0281_0002.pdb'),
 PosixPath('../../data/raw/megascale/pdb/2M8I.pdb'),
 PosixPath('../../data/raw/megascale/pdb/2WQG.pdb'),
 PosixPath('../../data/raw/megascale/pdb/r16_515_TrROS_Hall.pdb'),
 PosixPath('../../data/raw/megascale/pdb/EA:run3_1214_0004.pdb'),
 PosixPath('../../data/raw/megascale/pdb/6IPY.pdb'),
 PosixPath('../../data/raw/megascale/pdb/2RRU.pdb'),
 PosixPath('../../data/raw/megascale/pdb/r16_43_TrROS_Hall.pdb'),
 PosixPath('..

In [9]:
pdb_path = pdb_files[0]

In [10]:
import py3Dmol
pdb_text = pdb_path.read_text()

viewer = py3Dmol.view(width=800, height=600)
viewer.addModel(pdb_text, "pdb")
viewer.setStyle({"cartoon": {}})
viewer.zoomTo()
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Visualize mutations in red:

In [11]:
viewer = py3Dmol.view(width=800, height=600)
viewer.addModel(pdb_text, "pdb")

# Show the protein
viewer.setStyle({"cartoon": {}})

# Highlight residues 9 and 23
viewer.addStyle(
    {"resi": [9, 23]},
    {"stick": {}}
)

viewer.zoomTo()
viewer.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Building a simple model

Let's build a small model similar to ThermoMPNN-D that uses ProteinMPNN's hidden features as inputs.
Later, we will aggregate using the following: 

```python
self.attn_pool = nn.Sequential(
    nn.Linear(embedding_dim, 1)
)
weights = torch.softmax(self.attn_pool(combined), dim=1)  # [B, L, 1]
pooled = torch.sum(weights * combined, dim=1)  # [B
```